# Simple Linear Regression – Marketing ROI Analysis

## Project Overview
This project analyzes marketing spend data to identify which channel (TV, Radio, or Social Media) 
has the strongest impact on Sales, using Simple Linear Regression with Python's statsmodels library.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully!")

In [ ]:
df = pd.read_csv('marketing_and_sales_data_evaluate_lr.csv')
print("Shape:", df.shape)
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
df_clean = df.dropna()
print("Clean shape:", df_clean.shape)
print(df_clean.describe())

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,6))
numeric_df = df_clean[['TV','Radio','Social_Media','Sales']]
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.3f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5))
for i, col in enumerate(['TV','Radio','Social_Media']):
    axes[i].scatter(df_clean[col], df_clean['Sales'], alpha=0.6, color=['#2196F3','#4CAF50','#FF9800'][i])
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Sales')
    r = df_clean[col].corr(df_clean['Sales'])
    axes[i].set_title(f'{col} vs Sales (r={r:.3f})')
plt.suptitle('Marketing Channels vs Sales')
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df_clean[['TV','Radio','Social_Media','Sales']])
plt.suptitle('Pairplot', y=1.02)
plt.show()

## Variable Selection — Best Predictor

In [ ]:
correlations = df_clean[['TV','Radio','Social_Media']].corrwith(df_clean['Sales'])
print("Correlations with Sales:")
print(correlations.sort_values(ascending=False))
best_predictor = correlations.idxmax()
print(f"\nBest predictor: {best_predictor} (r={correlations[best_predictor]:.4f})")

## Model Building — OLS Simple Linear Regression

In [ ]:
X = sm.add_constant(df_clean[best_predictor])
y = df_clean['Sales']
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(df_clean[best_predictor], y, alpha=0.6, color='#2196F3', label='Actual Data')
plt.plot(df_clean[best_predictor], model.fittedvalues, color='red', linewidth=2.5, label='Regression Line')
plt.xlabel(f'{best_predictor} Budget')
plt.ylabel('Sales')
plt.title(f'Simple Linear Regression: {best_predictor} → Sales')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Assumption Checking — Diagnostic Plots

In [ ]:
residuals = model.resid
fitted = model.fittedvalues
fig, axes = plt.subplots(2,2,figsize=(12,10))

axes[0,0].scatter(fitted, residuals, alpha=0.6)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Residuals vs Fitted (Linearity/Homoscedasticity)')
axes[0,0].set_xlabel('Fitted Values')
axes[0,0].set_ylabel('Residuals')

stats.probplot(residuals, plot=axes[0,1])
axes[0,1].set_title('Q-Q Plot (Normality)')

axes[1,0].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.6, color='green')
axes[1,0].set_title('Scale-Location (Homoscedasticity)')
axes[1,0].set_xlabel('Fitted Values')

axes[1,1].hist(residuals, bins=20, color='orange', edgecolor='white')
axes[1,1].set_title('Residuals Histogram (Normality)')

plt.suptitle('OLS Assumption Diagnostics', fontsize=14)
plt.tight_layout()
plt.show()

## Results Interpretation & Business Recommendation

In [ ]:
intercept = model.params['const']
coef = model.params[best_predictor]
r2 = model.rsquared
pval = model.pvalues[best_predictor]

print("="*60)
print("     RESULTS INTERPRETATION & BUSINESS INSIGHTS")
print("="*60)
print(f"\nLinear Equation: Sales = {intercept:.4f} + {coef:.4f} x {best_predictor}")
print(f"\nR-squared: {r2:.4f}")
print(f"  => {r2*100:.1f}% of Sales variation explained by {best_predictor}")
print(f"\nCoefficient: {coef:.4f}")
print(f"  => $1 more in {best_predictor} => ${coef:.4f} more in Sales")
print(f"\nP-value: {pval:.6f}")
print(f"  => Statistically significant: {'YES' if pval < 0.05 else 'NO'}")
print(f"\nROI RECOMMENDATION:")
print(f"  Invest the most in {best_predictor} advertising.")
print(f"  It has the strongest proven impact on Sales.")
print("="*60)